# 03 · Silver con Change Data Feed

**Objetivo:** que silver se actualice **solo con lo que cambió** en bronze, usando el Change Data Feed (CDF) y `MERGE`.

    bronze_flights ──► table_changes(bronze) ──► limpiar y tipar ──► MERGE por vuelo_id ──► silver_vuelos

| Transformación | Antes (bronze) | Después (silver) |
|---|---|---|
| Id del vuelo | `icao24` + `firstSeen` | `vuelo_id` único |
| Horas | segundos Unix en UTC | `TIMESTAMP` en hora de Brasil |
| Callsign | `"TAM3456  "` | `"TAM3456"` + `aerolinea_icao = TAM` |
| Aeropuertos nulos | `null` | `"DESC"` (desconocido) |
| Duración | no existe | `duracion_min` |

In [0]:
%run ./00_Setup

## Paso 1 · Ver los cambios de bronze
`table_changes('tabla', versión_inicial)` devuelve cada fila que cambió con tres columnas extra:
`_change_type`, `_commit_version` y `_commit_timestamp`.

In [0]:
display(spark.sql(f"""
    SELECT _commit_version, _commit_timestamp, _change_type, count(*) AS filas
    FROM table_changes('{BRONZE}', 0)
    GROUP BY 1, 2, 3
    ORDER BY _commit_version, _change_type
"""))

👀 **Qué observar:** los vuelos nuevos son `insert`. Los vuelos vistos en un segundo aeropuerto aparecen como
`update_preimage` (antes) y `update_postimage` (después).

## Paso 2 · Ver la transformación
`to_silver()` (en `00_Setup`) convierte bronze en columnas limpias y con tipos correctos.

In [0]:
display(to_silver(spark.table(BRONZE)).limit(10))

## Paso 3 · Construir silver con CDF + MERGE
`build_silver()` hace cuatro cosas. Lee el código antes de ejecutarlo:

In [0]:
print(inspect.getsource(build_silver))

In [0]:
build_silver()
print("Vuelos en silver:", spark.table(SILVER).count())
display(spark.table(STATE))

👀 **Qué observar:** `pipeline_state` recuerda hasta qué versión de bronze se procesó.
La próxima vez, silver pedirá los cambios **desde la siguiente versión**.

## Paso 4 · Re-ejecutar: nada que hacer

In [0]:
build_silver()

## Paso 5 · Calidad de los datos en silver
Antes de pasar a gold, revisamos qué tan completos están los datos.

In [0]:
s = spark.table(SILVER)
total = s.count()

calidad = {
    "Vuelos totales":                   total,
    "Sin callsign":                     s.where("callsign IS NULL").count(),
    "Aerolínea OTROS (privados, etc.)": s.where("aerolinea_icao = 'OTROS'").count(),
    "Origen desconocido (DESC)":        s.where("origen_icao = 'DESC'").count(),
    "Destino desconocido (DESC)":       s.where("destino_icao = 'DESC'").count(),
    "Duración > 20 horas":              s.where("duracion_min > 1200").count(),
    "vuelo_id duplicados":              total - s.select("vuelo_id").distinct().count(),
    "Excluidos: sin hora de salida (bronze)": spark.table(BRONZE).where("firstSeen IS NULL").count(),
}
for regla, n in calidad.items():
    print(f"{regla:<35} {n:>7}  ({n / total:.1%})")

In [0]:
display(spark.table(BRONZE).where("firstSeen IS NULL")
        .withColumn("tipo", F.when(F.trim("callsign").rlike("^P[PRST][A-Z]{3}$"), "Matrícula brasileña (aviación general)")
                             .otherwise("Otro"))
        .groupBy("tipo", F.col("source_file").substr(9, 4).alias("aeropuerto"))
        .count().orderBy(F.col("count").desc()))

In [0]:
# Top aerolíneas y rutas: una primera mirada al negocio
display(s.groupBy("aerolinea_icao").count().orderBy(F.col("count").desc()).limit(10))
display(s.where("origen_icao <> 'DESC' AND destino_icao <> 'DESC'")
         .groupBy("origen_icao", "destino_icao").count()
         .orderBy(F.col("count").desc()).limit(10))

👀 **Qué observar:** `vuelo_id duplicados` debe ser **0**. Las aerolíneas principales deberían ser `TAM` (LATAM Brasil),
`GLO` (Gol) y `AZU` (Azul). Anota qué códigos aparecen: los usaremos en el catálogo de `dim_aerolinea`.

## Paso 6 · Un UPDATE y un DELETE que viajan por el CDF
Simulamos dos cambios en bronze: una corrección (otro aeropuerto de destino) y un borrado.
En el CDF un `UPDATE` produce dos filas (`update_preimage` y `update_postimage`) y un `DELETE`, una.

In [0]:
v_update, v_delete = [r for r in spark.table(BRONZE)
                                       .where("icao24 IS NOT NULL AND firstSeen IS NOT NULL AND lastSeen > firstSeen")
                                       .select("icao24", "firstSeen")
                                       .orderBy("firstSeen").limit(2).collect()]

spark.sql(f"""UPDATE {BRONZE} SET estArrivalAirport = 'SBGR'
              WHERE icao24 = '{v_update.icao24}' AND firstSeen = {v_update.firstSeen}""")
spark.sql(f"""DELETE FROM {BRONZE}
              WHERE icao24 = '{v_delete.icao24}' AND firstSeen = {v_delete.firstSeen}""")

id_update = f"{v_update.icao24}_{v_update.firstSeen}"
id_delete = f"{v_delete.icao24}_{v_delete.firstSeen}"
print("Actualizado:", id_update, "| Borrado:", id_delete)

In [0]:
nulos = spark.table(BRONZE).where("icao24 IS NULL OR firstSeen IS NULL")
print("Filas con clave nula en bronze:", nulos.count())
display(nulos.select("icao24", "firstSeen", "lastSeen", "callsign", "source_file", "_rescued_data").limit(10))

In [0]:
desde = last_processed("silver") + 1
display(read_changes(BRONZE, desde, table_version(BRONZE))
        .select("icao24", "firstSeen", "estArrivalAirport", "_change_type", "_commit_version")
        .orderBy("_commit_version"))

In [0]:
build_silver()

print("Destino en silver del vuelo actualizado:",
      spark.table(SILVER).where(F.col("vuelo_id") == id_update).select("destino_icao").first()[0])
print("¿Sigue en silver el vuelo borrado?:",
      spark.table(SILVER).where(F.col("vuelo_id") == id_delete).count() > 0)

👀 **Qué observar:** silver procesó **2 cambios**, no 15 mil filas. La imagen previa no se aplica,
la posterior actualiza la fila y el `delete` la elimina.

## Paso 7 · Los cambios de silver
Silver también tiene CDF activado: es lo que leerá gold en el siguiente notebook.

In [0]:
display(spark.sql(f"""
    SELECT _commit_version, _change_type, count(*) AS filas
    FROM table_changes('{SILVER}', 0)
    GROUP BY 1, 2
    ORDER BY 1, 2
"""))
show_history(SILVER)

## ✔ Checkpoint
- [ ] Sé leer `table_changes` y qué significa cada `_change_type`
- [ ] Silver se construyó con `MERGE` a partir de los cambios, no de toda la tabla
- [ ] `pipeline_state` guarda hasta qué versión se procesó
- [ ] `vuelo_id duplicados` = 0
- [ ] Un `UPDATE` y un `DELETE` en bronze llegaron a silver por el CDF

**Siguiente:** `04_Gold_Modelo_Estrella`